## VISp all-layer natural scene responses — visualisation
Author: patrick.mccarthy@dpag.ox.ac.uk

**Stimulus timing (from extraction script + data)**
| Event | Time |
|---|---|
| Stim onset | t = 0 ms |
| Stim offset | t ≈ 250 ms (stimulus duration ~250.2 ms) |
| Next stim onset | t ≈ 250 ms — **no ISI**, presentations are back-to-back |
| Recording window end | t = 350 ms |
| **Contamination warning** | **Data after ~250 ms is driven by the next (random) natural scene** |

Pre-stimulus baseline: none — `bin_edges` starts at 0 (stim onset).

In [ ]:
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = Path('/Users/pmccarthy/Documents/experimental_data/allen_visual_neuropixels_longwindow_5ms_bins/all_layers')
pkl_files = sorted(DATA_DIR.glob('*_alllayers_psth_responses.pkl'))
print(f'{len(pkl_files)} sessions found')

In [ ]:
# canonical layer order and colours
LAYER_ORDER  = ['VISp1', 'VISp2/3', 'VISp2/3a', 'VISp2/3b', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']
LAYER_COLORS = {
    'VISp1':    '#a6cee3',
    'VISp2/3':  '#1f78b4',
    'VISp2/3a': '#4393c3',
    'VISp2/3b': '#2166ac',
    'VISp4':    '#33a02c',
    'VISp5':    '#e31a1c',
    'VISp6a':   '#ff7f00',
    'VISp6b':   '#cab2d6',
}

# timing landmarks (seconds)
T_STIM_ONSET  = 0.0
T_STIM_OFFSET = 0.250   # stim duration ~250 ms
T_WIN_END     = 0.350   # recording window end

def shade_contamination(ax, alpha=0.12):
    """Grey shading over the post-offset contamination region."""
    ax.axvspan(T_STIM_OFFSET, T_WIN_END, color='grey', alpha=alpha, zorder=0)

def load_sessions(pkl_files):
    sessions = []
    for p in pkl_files:
        with open(p, 'rb') as f:
            sessions.append(pickle.load(f))
    return sessions

sessions = load_sessions(pkl_files)
print(f'Loaded {len(sessions)} sessions')
for s in sessions[:3]:
    layers, counts = np.unique(s['layer'], return_counts=True)
    print(f"  {s['session_id']}: " + ', '.join(f'{l}={c}' for l, c in zip(layers, counts)))

In [ ]:
def mean_psth_by_layer(session):
    """
    Returns:
        t            : bin centres (s), shape (num_bins,)
        layer_psth   : dict  layer_str -> mean FR (Hz), shape (num_bins,)
    Averaged over all stimuli, trials, and units within each layer.
    """
    responses = session['responses']          # (stim, trials, bins, units)
    layers    = np.array(session['layer'])    # (units,)
    bin_edges = session['bin_edges']
    bin_size  = float(np.diff(bin_edges[:2]))
    t = bin_edges[:-1] + bin_size / 2

    layer_psth = {}
    for lyr in np.unique(layers):
        mask = layers == lyr
        layer_psth[lyr] = responses[:, :, :, mask].mean(axis=(0, 1, 3)) / bin_size
    return t, layer_psth

### Overall PSTH — all VISp neurons (single session)

In [ ]:
SESSION_IDX = 0
s = sessions[SESSION_IDX]
bin_edges = s['bin_edges']
bin_size  = float(np.diff(bin_edges[:2]))
t = bin_edges[:-1] + bin_size / 2

# mean over all stim, trials, units -> (bins,)
overall_psth = s['responses'].mean(axis=(0, 1, 3)) / bin_size

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t * 1000, overall_psth, color='k', lw=1.8)
ax.axvline(T_STIM_ONSET  * 1000, color='tab:blue',  lw=1, ls='--', label='stim onset')
ax.axvline(T_STIM_OFFSET * 1000, color='tab:orange', lw=1, ls='--', label='stim offset / next onset')
ax.axvspan(T_STIM_OFFSET * 1000, T_WIN_END * 1000, color='grey', alpha=0.12,
           label='next-stim contamination', zorder=0)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f"Session {s['session_id']} — all VISp units, all stimuli")
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

### Overall PSTH — all VISp neurons, cross-session mean ± SEM

In [ ]:
all_overall = []
for s in sessions:
    bs = float(np.diff(s['bin_edges'][:2]))
    all_overall.append(s['responses'].mean(axis=(0, 1, 3)) / bs)

mu  = np.mean(all_overall, axis=0)
err = np.std(all_overall, axis=0) / np.sqrt(len(all_overall))
t_ms = t * 1000

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t_ms, mu, color='k', lw=1.8)
ax.fill_between(t_ms, mu - err, mu + err, alpha=0.25, color='k')
ax.axvline(T_STIM_ONSET  * 1000, color='tab:blue',  lw=1, ls='--', label='stim onset')
ax.axvline(T_STIM_OFFSET * 1000, color='tab:orange', lw=1, ls='--', label='stim offset / next onset')
ax.axvspan(T_STIM_OFFSET * 1000, T_WIN_END * 1000, color='grey', alpha=0.12,
           label='next-stim contamination', zorder=0)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'All VISp units — cross-session mean ± SEM (n={len(sessions)} sessions)')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

### Average PSTH per layer — single session (stacked)

In [ ]:
SESSION_IDX = 0
s = sessions[SESSION_IDX]
t, layer_psth = mean_psth_by_layer(s)
t_ms = t * 1000

layers_present = [l for l in LAYER_ORDER if l in layer_psth] + \
                 [l for l in layer_psth if l not in LAYER_ORDER]

fig, axes = plt.subplots(len(layers_present), 1,
                          figsize=(7, 2.2 * len(layers_present)),
                          sharex=True)
if len(layers_present) == 1:
    axes = [axes]

for ax, lyr in zip(axes, layers_present):
    color = LAYER_COLORS.get(lyr, '#888888')
    ax.plot(t_ms, layer_psth[lyr], color=color, lw=1.5)
    ax.axvline(T_STIM_ONSET  * 1000, color='tab:blue',  lw=0.8, ls='--')
    ax.axvline(T_STIM_OFFSET * 1000, color='tab:orange', lw=0.8, ls='--')
    ax.axvspan(T_STIM_OFFSET * 1000, T_WIN_END * 1000, color='grey', alpha=0.12, zorder=0)
    ax.set_ylabel('Hz', fontsize=8)
    ax.set_title(lyr, fontsize=9, loc='left', pad=2)
    ax.spines[['top', 'right']].set_visible(False)

axes[-1].set_xlabel('Time from stim onset (ms)')
fig.suptitle(f"Session {s['session_id']} — mean PSTH by layer", y=1.01)
fig.tight_layout()
plt.show()

### Overlay all layers on one axis (single session)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for lyr in layers_present:
    color = LAYER_COLORS.get(lyr, '#888888')
    ax.plot(t_ms, layer_psth[lyr], label=lyr, color=color, lw=1.5)

ax.axvline(T_STIM_ONSET  * 1000, color='tab:blue',  lw=0.9, ls='--', label='stim onset')
ax.axvline(T_STIM_OFFSET * 1000, color='tab:orange', lw=0.9, ls='--', label='stim offset')
ax.axvspan(T_STIM_OFFSET * 1000, T_WIN_END * 1000, color='grey', alpha=0.12,
           label='contamination', zorder=0)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f"Session {s['session_id']} — all layers overlaid")
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

### Cross-session average PSTH per layer (mean ± SEM)

In [ ]:
layer_traces = defaultdict(list)

for s in sessions:
    t, lp = mean_psth_by_layer(s)
    for lyr, trace in lp.items():
        layer_traces[lyr].append(trace)

t_ms = t * 1000

print('Sessions per layer:')
for lyr in LAYER_ORDER:
    if lyr in layer_traces:
        print(f'  {lyr}: {len(layer_traces[lyr])} sessions')

In [ ]:
all_layers = [l for l in LAYER_ORDER if l in layer_traces] + \
             [l for l in layer_traces if l not in LAYER_ORDER]

fig, ax = plt.subplots(figsize=(7, 4))

for lyr in all_layers:
    traces = layer_traces[lyr]
    mu  = np.mean(traces, axis=0)
    err = np.std(traces, axis=0) / np.sqrt(len(traces))
    color = LAYER_COLORS.get(lyr, '#888888')
    ax.plot(t_ms, mu, label=f'{lyr} (n={len(traces)})', color=color, lw=1.8)
    ax.fill_between(t_ms, mu - err, mu + err, alpha=0.2, color=color)

ax.axvline(T_STIM_ONSET  * 1000, color='tab:blue',  lw=0.9, ls='--', label='stim onset')
ax.axvline(T_STIM_OFFSET * 1000, color='tab:orange', lw=0.9, ls='--', label='stim offset')
ax.axvspan(T_STIM_OFFSET * 1000, T_WIN_END * 1000, color='grey', alpha=0.12,
           label='contamination', zorder=0)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Cross-session mean ± SEM — all VISp layers')
ax.legend(frameon=False, fontsize=8, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

### Heatmap: stimulus × time for each layer (single session)

In [ ]:
SESSION_IDX = 0
s = sessions[SESSION_IDX]
bin_size  = float(np.diff(s['bin_edges'][:2]))
responses = s['responses']
layers    = np.array(s['layer'])
t_ms      = s['bin_edges'][:-1] * 1000 + bin_size * 500  # bin centres in ms

layers_present = [l for l in LAYER_ORDER if l in np.unique(layers)] + \
                 [l for l in np.unique(layers) if l not in LAYER_ORDER]

fig, axes = plt.subplots(1, len(layers_present),
                          figsize=(4 * len(layers_present), 5),
                          sharey=True)
if len(layers_present) == 1:
    axes = [axes]

for ax, lyr in zip(axes, layers_present):
    mask = layers == lyr
    heatmap = responses[:, :, :, mask].mean(axis=(1, 3)) / bin_size  # (stim, bins) in Hz
    sort_idx = np.argsort(np.argmax(heatmap, axis=1))
    im = ax.imshow(heatmap[sort_idx],
                   aspect='auto', origin='upper',
                   extent=[t_ms[0], t_ms[-1], heatmap.shape[0], 0],
                   cmap='inferno')
    ax.axvline(T_STIM_ONSET  * 1000, color='w', lw=0.9, ls='--')
    ax.axvline(T_STIM_OFFSET * 1000, color='cyan', lw=0.9, ls='--')
    ax.set_title(lyr, fontsize=9)
    ax.set_xlabel('Time (ms)', fontsize=8)
    fig.colorbar(im, ax=ax, label='Hz', shrink=0.6)

axes[0].set_ylabel('Stimulus (sorted by peak time)')
fig.suptitle(f"Session {s['session_id']} — stimulus × time heatmaps per layer\n"
             "(cyan dashed = stim offset / next onset)")
fig.tight_layout()
plt.show()

### Peak response amplitude and latency by layer (cross-session)

In [ ]:
records = []
for s in sessions:
    t, lp = mean_psth_by_layer(s)
    for lyr, trace in lp.items():
        # restrict to stimulus-driven window (0 to offset) to avoid contamination
        stim_mask = t <= T_STIM_OFFSET
        peak_idx  = np.argmax(trace[stim_mask])
        records.append({
            'session_id':  s['session_id'],
            'layer':       lyr,
            'peak_hz':     trace[stim_mask][peak_idx],
            'peak_lat_ms': t[stim_mask][peak_idx] * 1000,
        })

df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for metric, ax, ylabel in [
    ('peak_hz',     axes[0], 'Peak firing rate (Hz)'),
    ('peak_lat_ms', axes[1], 'Peak latency (ms)'),
]:
    plot_layers = [l for l in LAYER_ORDER if l in df['layer'].values] + \
                  [l for l in df['layer'].unique() if l not in LAYER_ORDER]
    means  = [df[df.layer == l][metric].mean() for l in plot_layers]
    errors = [df[df.layer == l][metric].sem()  for l in plot_layers]
    colors = [LAYER_COLORS.get(l, '#888888') for l in plot_layers]

    ax.bar(plot_layers, means, yerr=errors, color=colors, capsize=4, edgecolor='k', linewidth=0.6)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Layer')
    ax.tick_params(axis='x', rotation=30)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Peak response amplitude and latency by layer (cross-session mean ± SEM)\n'
             'Peak computed within stim-driven window only (0–250 ms)')
fig.tight_layout()
plt.show()

df.groupby('layer')[['peak_hz', 'peak_lat_ms']].agg(['mean', 'sem'])